# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the FAIR^2 colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data elements are referenced by their Croissant schema `@id` fields, allowing unambiguous dataset navigation and processing.

### Dataset Source
The dataset is described by a Croissant schema, available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
It contains detailed clinicopathological, demographic, anatomical, and molecular features for 77 colorectal cancer survivor cases.

In [ ]:
# Ensure `mlcroissant` is available (uncomment if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the colorectal cancer survivor dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dictionary

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Let's examine all available RecordSets, their `@id`s, and the fields belonging to each RecordSet. We use Croissant `@id` fields for reference and access.

In [ ]:
# List all record sets and their fields by @id
print("Available RecordSets and their fields:")
for record_set in dataset.record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '-')}")
    print(f"  Fields:")
    for field in record_set.fields:
        field_type_name = getattr(field.data_type, 'name', field.data_type) if hasattr(field, 'data_type') and field.data_type else '-' 
        print(f"    - @id: {field.id} | Name: {field.name} | Data type: {field_type_name}")

## 3. Data Extraction
We will extract the data for **each RecordSet** using their `@id` fields, and load them into Pandas DataFrames.

*For demonstration, the primary RecordSet containing the main tabular data is typically the one named like `second_primary_crc`, but let's discover it dynamically and extract all.*

In [ ]:
# Discover all RecordSets and load them into DataFrames by their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df

print("\nLoaded DataFrames (non-empty):")
for rs_id, df in dataframes.items():
    if not df.empty:
        print(f"- RecordSet {rs_id}: {df.shape[0]} rows, columns: {df.columns.tolist()}")

For example, suppose one main record set is:

```python
main_rs_id = record_set_ids[0]  # (adjust to your dataset, e.g. the first or the one with main fields)
```
Let's preview the first few rows and columns.

In [ ]:
# Choose a primary record set to analyze in detail
# Here, we select the RecordSet with non-empty DataFrame and most columns
main_rs_id = max([(rs_id, df.shape[1]) for rs_id, df in dataframes.items() if not df.empty], key=lambda x: x[1])[0]

print(f"\nPrimary RecordSet selected: {main_rs_id}")
print("Columns:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field, filter records with values above a threshold, normalize it, and optionally group by a categorical field.

> **All fields and columns are referenced by `@id`.**

Let's list all numeric fields' `@id`s for possible selection.

In [ ]:
# List numeric fields in the main RecordSet for EDA
main_rs = None
for rs in dataset.record_sets:
    if rs.id == main_rs_id:
        main_rs = rs
        break

numeric_types = {'Integer', 'Float', 'Number'}
numeric_fields = [f for f in main_rs.fields if hasattr(f, 'data_type') and (getattr(f.data_type, 'name', None) in numeric_types or f.data_type in numeric_types)]
print("Numeric fields in RecordSet:")
for f in numeric_fields:
    print(f"- {f.id} (name: {f.name})")

***
Suppose a numeric field, e.g. age or interval field with `@id` found above, is extracted for further EDA. Let's proceed with filtering, normalization, and grouping using these IDs.

In [ ]:
# Example: Select one numeric field @id (replace with correct one based on previous output)
if numeric_fields:
    numeric_field_id = numeric_fields[0].id  # First numeric field found
    print(f"Selected numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric fields found.")

# Filtering threshold (you may want to adapt this depending on field semantics)
threshold = 10
df = dataframes[main_rs_id]

# Ensure the numeric field column exists and is numeric
if numeric_field_id in df.columns:
    # Convert to numeric dtype if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold].copy()

    print(f"Filtered {len(filtered_df)} rows with {numeric_field_id} > {threshold}")

    # Normalize the selected field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Try grouping by a non-numeric categorical field, if available
    group_field = None
    for f in main_rs.fields:
        dt = getattr(f.data_type, 'name', f.data_type)
        if dt == 'Text' and f.id in filtered_df.columns:
            group_field = f.id
            break

    if group_field:
        print(f"\nGrouping by field: {group_field}")
        grouped_stats = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(grouped_stats.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print(f"Field {numeric_field_id} not present in main dataframe columns.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and explore relationship with the chosen group field (if available).

We use Matplotlib for quick plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field after filtering
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group, if applicable
if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated end-to-end loading and exploration of a Croissant-described clinical dataset using `mlcroissant`, always referencing data by their unique `@id`s.

- We loaded metadata and all record sets using the Croissant API.
- All data extractions, analyses, and transformations referenced fields and record sets by `@id`.
- We performed basic EDA including numeric column filtering, normalization, and grouping, and visualized distributions.

You can extend this workflow by referencing additional record sets and their `@id`s, or building advanced analyses atop this reproducible, schema-driven workflow.